# 第十四章：综合模拟交易台 / Chapter 14: Integrated Simulation Desk

## 你学完了全部内容! 现在用一个面板把它们串起来。
## You have finished the whole tutorial! Let's stitch every concept together on one panel.

> 选择任意产品 → 设置参数 → 实时看到：
> - 盈亏曲线
> - 预付款比例
> - 风险等级
> - 7种情景分析
>
> Pick any product → set your parameters → watch in real time:
> - P&L curve
> - Margin level
> - Risk rating
> - Seven built-in scenario cases

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, ToggleButtons
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done!')

## 开始交易! / Start Trading!

> 1. 选一个产品（外汇、黄金、比特币...）
> 2. 设置你的资金和杠杆
> 3. 选择做多还是做空
> 4. 观察四个面板的实时变化
>
> 1. Pick a product (FX, gold, BTC, ...)
> 2. Set your capital and leverage
> 3. Choose long or short
> 4. Watch the four panels update live

In [ ]:
PRODUCTS = {
    'EURUSD':   {'contract': 100000, 'tick': 0.00001, 'ref_price': 1.088},
    'XAUUSD (Gold)':  {'contract': 100, 'tick': 0.01, 'ref_price': 2300},
    'BTCUSD':   {'contract': 1, 'tick': 0.01, 'ref_price': 63000},
    'US30':     {'contract': 1, 'tick': 0.01, 'ref_price': 39000},
    'AAPL':     {'contract': 100, 'tick': 0.01, 'ref_price': 180},
}


def trading_desk(product='XAUUSD (Gold)', capital=10000, leverage=100, lots=1.0,
                 side='Long (buy)'):
    plt.close('all')
    cfg = PRODUCTS[product]
    price = cfg['ref_price']
    contract = cfg['contract']
    is_long = 'Long' in side

    margin = lots * contract * price / leverage
    pip_value = cfg['tick'] * contract * lots
    margin_ratio = margin / capital * 100
    if is_long:
        liq_price = price - (capital - margin * 0.30) / (lots * contract)
    else:
        liq_price = price + (capital - margin * 0.30) / (lots * contract)

    fig = plt.figure(figsize=(14, 8))
    p_range = np.linspace(price * 0.7, price * 1.3, 300)
    if is_long:
        pnl = (p_range - price) * lots * contract
    else:
        pnl = (price - p_range) * lots * contract
    margin_level = ((capital + pnl) / margin) * 100

    # P&L curve
    ax1 = fig.add_subplot(221)
    ax1.fill_between(p_range, pnl, 0, where=(pnl >= 0), color=C_GREEN, alpha=0.2)
    ax1.fill_between(p_range, pnl, 0, where=(pnl < 0), color=C_RED, alpha=0.2)
    ax1.plot(p_range, pnl, 'k-', lw=2)
    ax1.axhline(y=0, color='gray', ls='--')
    if 0 < liq_price < price * 2:
        ax1.axvline(x=liq_price, color=C_RED, ls='--', alpha=0.7)
        ax1.annotate(f'Liq {liq_price:.2f}', xy=(liq_price, 0), fontsize=9, color=C_RED, fontweight='bold',
                     xytext=(liq_price, pnl.min() * 0.4),
                     arrowprops=dict(arrowstyle='->', color=C_RED))
    side_label = 'Long' if is_long else 'Short'
    ax1.set_xlabel('Price'); ax1.set_ylabel('P&L (USD)')
    ax1.set_title(f'{product} {side_label} P&L', fontweight='bold')
    ax1.grid(True, alpha=0.2)

    # Margin level
    ax2 = fig.add_subplot(222)
    ax2.plot(p_range, margin_level, color=C_BLUE, lw=2)
    ax2.axhline(y=100, color=C_ORANGE, ls='--', label='100% warning')
    ax2.axhline(y=30, color=C_RED, ls='--', label='30% stopout')
    ax2.fill_between(p_range, 0, margin_level, where=(margin_level <= 30), color=C_RED, alpha=0.2)
    ax2.fill_between(p_range, 0, margin_level, where=((margin_level > 30) & (margin_level <= 100)),
                     color=C_ORANGE, alpha=0.1)
    ax2.set_xlabel('Price'); ax2.set_ylabel('Margin Level (%)')
    ax2.set_title('Margin Level', fontweight='bold'); ax2.legend(fontsize=8)
    ax2.set_ylim(0, min(300, margin_level.max() * 1.1))
    ax2.grid(True, alpha=0.2)

    # Parameters panel
    ax3 = fig.add_subplot(223)
    ax3.set_xlim(0, 10); ax3.set_ylim(0, 10); ax3.axis('off')
    ax3.set_title('Your Trade', fontweight='bold')
    risk_color = C_GREEN if margin_ratio < 30 else C_ORANGE if margin_ratio < 60 else C_RED
    risk_text = 'LOW RISK' if margin_ratio < 30 else 'CAUTION' if margin_ratio < 60 else 'HIGH RISK!'
    info = [f'{product} | {side_label}',
            f'Margin: ${margin:,.2f} ({margin_ratio:.1f}%)',
            f'Pip P&L: ${pip_value:.4f}',
            f'Liq price: {liq_price:.4f}',
            f'Distance: {abs(price - liq_price):.4f} ({abs(price - liq_price) / price * 100:.1f}%)']
    for i, txt in enumerate(info):
        bg = C_BLUE if i == 0 else 'lightyellow'
        fc = 'white' if i == 0 else 'black'
        ax3.text(5, 8.5 - i * 1.3, txt, ha='center', fontsize=10, color=fc,
                 fontweight='bold' if i == 0 else 'normal',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor=bg,
                          edgecolor='lightgray', alpha=0.8 if i > 0 else 0.3))
    ax3.text(5, 1.5, risk_text, ha='center', fontsize=20, fontweight='bold', color=risk_color,
             bbox=dict(boxstyle='round,pad=0.5', facecolor=risk_color, alpha=0.15))

    # Scenario analysis
    ax4 = fig.add_subplot(224)
    ax4.set_xlim(0, 10); ax4.set_ylim(0, 10); ax4.axis('off')
    ax4.set_title('Scenarios', fontweight='bold')
    scenes = [(-5, 'Crash -5%'), (-2, 'Down -2%'), (-1, 'Down -1%'),
              (0, 'Flat'),
              (1, 'Up +1%'), (2, 'Up +2%'), (5, 'Rally +5%')]
    ax4.text(1.5, 9.3, 'Scenario', fontsize=9, fontweight='bold')
    ax4.text(5, 9.3, 'P&L', fontsize=9, fontweight='bold')
    ax4.text(8.5, 9.3, 'ROI', fontsize=9, fontweight='bold')
    ax4.plot([0.3, 9.7], [9.0, 9.0], color='gray', lw=0.5)
    for i, (pct, name) in enumerate(scenes):
        new_p = price * (1 + pct / 100)
        sc_pnl = (new_p - price) * lots * contract if is_long else (price - new_p) * lots * contract
        col = C_GREEN if sc_pnl >= 0 else C_RED
        y = 8.3 - i * 1.15
        ax4.text(1.5, y, name, fontsize=9)
        ax4.text(5, y, f'${sc_pnl:+,.0f}', fontsize=9, fontweight='bold', color=col, ha='center')
        ax4.text(8.5, y, f'{sc_pnl / margin * 100:+.1f}%', fontsize=9, fontweight='bold', color=col, ha='center')

    plt.tight_layout()
    plt.show()

interact(trading_desk,
         product=Dropdown(options=list(PRODUCTS.keys()), value='XAUUSD (Gold)', description='Product:'),
         capital=IntSlider(value=10000, min=500, max=100000, step=500, description='Capital($):'),
         leverage=IntSlider(value=100, min=2, max=500, step=10, description='Leverage:'),
         lots=FloatSlider(value=1.0, min=0.01, max=20, step=0.1, description='Lots:'),
         side=ToggleButtons(options=['Long (buy)', 'Short (sell)'], description='Side:'));

---

# 恭喜你! 教程全部完成! / Congratulations! You Finished the Whole Tutorial!

## 你学到了什么 / What You Have Learned

| 章节 / Chapter | 内容 / Topic |
|------|------|
| Ch01 | 金融市场总览：现货 vs 衍生品 vs 杠杆 / Financial-market overview: spot vs. derivatives vs. leverage |
| Ch02 | 保证金和爆仓：核心风控 / Margin & liquidation: the core of risk control |
| Ch03 | 交易成本：点差、手续费、隔夜利息 / Trading costs: spread, commission, swap |
| Ch04 | 远期和期货：无套利定价 / Forwards & futures: no-arbitrage pricing |
| Ch05 | 期权基础：Call/Put/买方/卖方 / Option basics: calls, puts, buyers, sellers |
| Ch06 | Black-Scholes 定价 / Black-Scholes pricing |
| Ch07 | 希腊字母：期权的仪表盘 / The Greeks: the option's dashboard |
| Ch08 | 波动率微笑：现实为什么不完美 / Vol smile: why reality does not follow the textbook |
| Ch09 | 期权策略：像搭积木一样拼期权 / Option strategies: LEGO for options |
| Ch10 | 利率互换：交换风险 / Interest-rate swaps: exchanging risk |
| Ch11 | 永续合约：加密货币的发明 / Perpetual swaps: a crypto invention |
| Ch12 | U本位 vs 币本位：凸性风险 / USDT-margined vs. coin-margined: convexity risk |
| Ch13 | 极端行情：杠杆的生死线 / Crash simulations: leverage's do-or-die line |
| Ch14 | 综合交易台：全部整合 / Integrated desk: everything together |

## 最重要的三句话 / The Three Most Important Takeaways

1. **杠杆是双刃剑** — 放大盈利也放大亏损
   **Leverage is a double-edged sword** — it magnifies gains *and* losses.
2. **风险管理比赚钱更重要** — 活下来才能赚钱
   **Risk management matters more than profit** — you have to survive before you can win.
3. **不懂的产品不要碰** — 先学习，再实战
   **Never trade what you do not understand** — study first, trade later.